# Transformer inference
1. Data: the English-French pairs and the test sentences of the encoder-decoder Note.
2. A small transformer (2 encoder + 2 decoder blocks, d_model = 128), trained with teacher forcing.
3. Greedy inference: the encoder runs once, the decoder once per output word.
4. Checks: the step-by-step outputs equal one teacher-forced pass over the same words; earlier positions do not
   change from step to step; what happens without the mask at inference.
5. Data for the figures and for the cross-attention Note.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import re, time, json, random, collections
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

import warnings
warnings.filterwarnings("ignore", message="You are using a softmax over axis")   # step 1: one key, weight 1
np.set_printoptions(precision=3, suppress=True)
tf.config.experimental.enable_tensor_float_32_execution(False)   # exact float32 matmuls, for the equality checks
print(tf.config.list_physical_devices("GPU"))

I0000 00:00:1791082373.966733 2146276 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Data
Same corpus, filter, split and vocabularies as the encoder-decoder Note: English up to 8 words, French up to
10; 1,500 held-out English sentences for testing; 40,000 training pairs.

In [2]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = open(os.path.join(path, "fra.txt"), encoding="utf8").read().strip().split("\n")
tok = lambda s: re.findall(r"[\w']+|[^\w\s]", s.lower())
groups = collections.defaultdict(list)
for line in lines:
    e, f = line.split("\t")[:2]
    e, f = tok(e), tok(f)
    if len(e) <= 8 and len(f) <= 10:
        groups[tuple(e)].append(f)
keys = sorted(groups)
random.Random(0).shuffle(keys)
test_keys, train_keys = keys[:1500], keys[1500:]
train = [(list(e), f) for e in train_keys for f in groups[e]]
random.Random(1).shuffle(train)
train = train[:40000]


def vocab(seqs, n):
    c = collections.Counter(w for s in seqs for w in s)
    words = ["<pad>", "<unk>", "<start>", "<end>"] + [w for w, _ in c.most_common(n)]
    return {w: i for i, w in enumerate(words)}, words


en_ix, en_w = vocab([e for e, _ in train], 6000)
fr_ix, fr_w = vocab([f for _, f in train], 8000)
LE, LF = 8, 12
START, END = fr_ix["<start>"], fr_ix["<end>"]


def encode(seqs, ix, L, add=False):
    X = np.zeros((len(seqs), L), "int32")
    for i, s in enumerate(seqs):
        s = ["<start>"] + s + ["<end>"] if add else s
        ids = [ix.get(w, 1) for w in s][:L]
        X[i, :len(ids)] = ids
    return X


Xe = encode([e for e, _ in train], en_ix, LE)
Yf = encode([f for _, f in train], fr_ix, LF, add=True)
Xd, Yd = Yf[:, :-1], Yf[:, 1:]                    # decoder input: <start> + words; target: words + <end>
test_en = [list(k) for k in test_keys]
test_refs = [groups[k] for k in test_keys]
print(len(train), "training pairs,", len(test_en), "test sentences; vocabularies", len(en_w), len(fr_w))


def ngrams(s, n):
    return collections.Counter(tuple(s[i:i + n]) for i in range(len(s) - n + 1))


def bleu(hyps, refss):
    """Corpus BLEU (4-grams, brevity penalty), all references of a sentence used; same code as the encoder-decoder Note."""
    match, total = np.zeros(4), np.zeros(4)
    hyp_len = ref_len = 0
    for h, refs in zip(hyps, refss):
        hyp_len += len(h)
        ref_len += min((abs(len(r) - len(h)), len(r)) for r in refs)[1]
        for n in range(1, 5):
            hc = ngrams(h, n)
            best = collections.Counter()
            for r in refs:
                best |= ngrams(r, n)
            match[n - 1] += sum(min(c, best[g]) for g, c in hc.items())
            total[n - 1] += max(len(h) - n + 1, 0)
    if match.min() == 0:
        return 0.0
    bp = 1.0 if hyp_len > ref_len else np.exp(1 - ref_len / max(hyp_len, 1))
    return 100 * bp * np.exp(np.mean(np.log(match / total)))

40000 training pairs, 1500 test sentences; vocabularies 6004 8004


## 2. A small transformer
The paper's design (post-norm blocks, sinusoidal positions, embeddings times sqrt(d_model), dropout 0.1), smaller:
d_model = 128, 4 heads, d_ff = 512, 2 encoder and 2 decoder blocks.

In [3]:
D, HEADS, DFF, NBLOCKS = 128, 4, 512, 2


def positional_encoding(n_pos, d_model):
    pos = np.arange(n_pos)[:, None]
    i = np.arange(d_model // 2)[None, :]
    angle = pos / 10000 ** (2 * i / d_model)
    pe = np.zeros((n_pos, d_model))
    pe[:, 0::2], pe[:, 1::2] = np.sin(angle), np.cos(angle)
    return pe.astype("float32")


def pad_mask(q_ids, k_ids):
    """(batch, queries, keys) boolean: True where the key is a real token, not padding."""
    return tf.logical_and(tf.ones_like(q_ids, tf.bool)[:, :, None], (k_ids > 0)[:, None, :])


class EncoderBlock(keras.layers.Layer):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.mha = keras.layers.MultiHeadAttention(HEADS, D // HEADS, dropout=0.1)
        self.n1, self.n2 = keras.layers.LayerNormalization(), keras.layers.LayerNormalization()
        self.f1, self.f2 = keras.layers.Dense(DFF, activation="relu"), keras.layers.Dense(D)
        self.drop = keras.layers.Dropout(0.1)

    def call(self, x, mask, training=False):
        x = self.n1(x + self.drop(self.mha(x, x, attention_mask=mask, training=training), training=training))
        return self.n2(x + self.drop(self.f2(self.f1(x)), training=training))


class DecoderBlock(keras.layers.Layer):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.self_mha = keras.layers.MultiHeadAttention(HEADS, D // HEADS, dropout=0.1)
        self.cross_mha = keras.layers.MultiHeadAttention(HEADS, D // HEADS, dropout=0.1)
        self.n1, self.n2, self.n3 = [keras.layers.LayerNormalization() for _ in range(3)]
        self.f1, self.f2 = keras.layers.Dense(DFF, activation="relu"), keras.layers.Dense(D)
        self.drop = keras.layers.Dropout(0.1)

    def call(self, y, enc, cross_mask, causal=True, training=False):
        z = self.self_mha(y, y, use_causal_mask=causal, training=training)
        y = self.n1(y + self.drop(z, training=training))
        zc, w = self.cross_mha(y, enc, attention_mask=cross_mask, return_attention_scores=True, training=training)
        y = self.n2(y + self.drop(zc, training=training))
        return self.n3(y + self.drop(self.f2(self.f1(y)), training=training)), w


class Transformer(keras.Model):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.emb_en = keras.layers.Embedding(len(en_w), D)
        self.emb_fr = keras.layers.Embedding(len(fr_w), D)
        self.pe = tf.constant(positional_encoding(64, D))
        self.enc_blocks = [EncoderBlock() for _ in range(NBLOCKS)]
        self.dec_blocks = [DecoderBlock() for _ in range(NBLOCKS)]
        self.drop = keras.layers.Dropout(0.1)
        self.out = keras.layers.Dense(len(fr_w))          # linear layer; the softmax is applied in the loss / below

    def embed(self, emb, ids, training):
        n = tf.shape(ids)[1]
        return self.drop(emb(ids) * np.sqrt(D).astype("float32") + self.pe[:n], training=training)

    def encode(self, src, training=False):
        x = self.embed(self.emb_en, src, training)
        mask = pad_mask(src, src)
        for b in self.enc_blocks:
            x = b(x, mask, training=training)
        return x

    def decode(self, tgt, enc, src, causal=True, training=False, last_only=False):
        y = self.embed(self.emb_fr, tgt, training)
        cross = []
        for b in self.dec_blocks:
            y, w = b(y, enc, pad_mask(tgt, src), causal=causal, training=training)
            cross.append(w)
        return self.out(y[:, -1:] if last_only else y), y, cross   # last_only: inference needs only the last position

    def call(self, inputs, training=False):
        src, tgt = inputs
        return self.decode(tgt, self.encode(src, training), src, training=training)[0]


class PaperSchedule(keras.optimizers.schedules.LearningRateSchedule):
    """lrate = d_model^-0.5 * min(step^-0.5, step * warmup^-1.5) (Vaswani et al. 2017, eq. 3)."""
    def __init__(self, warmup):
        self.warmup = warmup

    def __call__(self, step):
        step = tf.cast(step + 1, tf.float32)
        return D ** -0.5 * tf.minimum(step ** -0.5, step * self.warmup ** -1.5)


def build(seed):
    keras.utils.set_random_seed(seed)
    m = Transformer()
    m((Xe[:2], Xd[:2]))
    m.compile(optimizer=keras.optimizers.Adam(PaperSchedule(2000), beta_1=0.9, beta_2=0.98, epsilon=1e-9),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True, ignore_class=0),
              metrics=[keras.metrics.SparseCategoricalAccuracy()])
    return m


model = build(0)
print("parameters:", model.count_params())

parameters: 3751236


## 3. Training (teacher forcing: all positions of the shifted French sentence at once)

In [4]:
t0 = time.time()
hist = model.fit((Xe, Xd), Yd, batch_size=128, epochs=25, validation_split=0.05, verbose=2).history
print("training time (s):", round(time.time() - t0))
pd.DataFrame(hist).rename_axis("epoch").to_csv("data/training_history.csv")

Epoch 1/25


297/297 - 51s - 171ms/step - loss: 7.0969 - sparse_categorical_accuracy: 0.1241 - val_loss: 4.9577 - val_sparse_categorical_accuracy: 0.2157


Epoch 2/25


297/297 - 8s - 27ms/step - loss: 4.0793 - sparse_categorical_accuracy: 0.2725 - val_loss: 3.2978 - val_sparse_categorical_accuracy: 0.3281


Epoch 3/25


...

Epoch 24/25


297/297 - 8s - 29ms/step - loss: 0.3703 - sparse_categorical_accuracy: 0.6203 - val_loss: 1.5401 - val_sparse_categorical_accuracy: 0.5248


Epoch 25/25


297/297 - 8s - 28ms/step - loss: 0.3561 - sparse_categorical_accuracy: 0.6230 - val_loss: 1.5297 - val_sparse_categorical_accuracy: 0.5261


training time (s): 250


## 4. Greedy inference
The encoder runs once per sentence. The decoder runs once per output word: at step t its input is <start> plus
the t - 1 words chosen so far; only the last position's probabilities choose the next word.

In [5]:
calls = collections.Counter()


def translate(sentences, causal=True, keep=False):
    src = encode(sentences, en_ix, LE)
    enc = model.encode(src)                                      # once
    calls["encoder"] += 1
    tgt = np.full((len(sentences), 1), START, "int32")
    done = np.zeros(len(sentences), bool)
    steps = []
    for t in range(LF - 1):
        logits, _, cross = model.decode(tgt, enc, src, causal=causal, last_only=True)   # once per output word
        calls["decoder"] += 1
        p = tf.nn.softmax(logits[:, -1]).numpy()                 # last position only
        nxt = p.argmax(-1).astype("int32")
        nxt[done] = 0                                            # finished sentences get padding
        if keep:
            steps.append(dict(p=p[0], cross=cross[-1].numpy()[0, :, -1, :]))   # last block, all heads, last row
        tgt = np.concatenate([tgt, nxt[:, None]], 1)
        done |= nxt == END
        if done.all():
            break
    out = []
    for row in tgt[:, 1:]:
        words = []
        for k in row:
            if k in (END, 0):
                break
            words.append(fr_w[k])
        out.append(words)
    return (out, tgt, steps) if keep else out


def translate_all(sentences, causal=True, chunk=250):
    """Greedy decoding in chunks of 250 sentences (the GPU is shared)."""
    return [h for i in range(0, len(sentences), chunk) for h in translate(sentences[i:i + chunk], causal=causal)]


calls.clear()
t0 = time.time()
hyp = translate_all(test_en)
print(len(test_en), "test sentences in", round(time.time() - t0, 1), "s, in 6 chunks of 250; calls:", dict(calls))
print("test BLEU:", round(bleu(hyp, test_refs), 1))
rows = []
for e, h, r in list(zip(test_en, hyp, test_refs))[:12]:
    print(" ".join(e), "->", " ".join(h), "   | reference:", " ".join(r[0]))
pd.DataFrame(dict(english=[" ".join(e) for e in test_en[:30]], predicted=[" ".join(h) for h in hyp[:30]],
                  reference=[" ".join(r[0]) for r in test_refs[:30]])).to_csv("data/translations.csv", index=False)

1500 test sentences in 4.4 s, in 6 chunks of 250; calls: {'encoder': 6, 'decoder': 66}
test BLEU: 41.4
that doesn't help me . -> ça ne m'aide pas .    | reference: ça ne m'aide pas .
i like your house . -> j'aime votre maison .    | reference: j'aime votre maison .
that guy is completely nuts ! -> ce mec est complètement <unk> !    | reference: ce mec est complètement zinzin !
i'm sorry , but i don't understand . -> je suis désolé , mais je ne comprends pas .    | reference: je suis désolé mais je ne comprends pas .
she accompanied her friend to the concert . -> elle a accompagné de son ami .    | reference: elle accompagna son ami au concert .
we have an unusual relationship . -> nous avons une relation avec les relation .    | reference: nous avons une relation inhabituelle .
what's his most recent novel ? -> qu'est - ce qui a le plus de <unk> ?    | reference: quel est son roman le plus récent ?
the pain has mostly gone away . -> la douleur est partie de suite .    | reference: la d

### One sentence, step by step: the 5 most likely words at each step and the cross-attention of the last block

In [6]:
step_rows, cross_rows = [], []
for demo in ("we're friends .", "i think you're right ."):
    calls.clear()
    out, tgt, steps = translate([demo.split()], keep=True)
    print(demo, "->", " ".join(out[0]), "  calls:", dict(calls))
    for t, s in enumerate(steps):
        top = s["p"].argsort()[::-1][:5]
        print(" step", t + 1, "input:", [fr_w[k] for k in tgt[0, :t + 1]], "->",
              [(fr_w[k], round(float(s["p"][k]), 3)) for k in top])
        for rank, k in enumerate(top):
            step_rows.append(dict(sentence=demo, step=t + 1, rank=rank + 1, word=fr_w[k], prob=float(s["p"][k])))
        for j, w in enumerate(demo.split()):
            cross_rows.append(dict(sentence=demo, step=t + 1, english=w, weight=float(s["cross"][:, j].mean())))
pd.DataFrame(step_rows).round(4).to_csv("data/decode_steps.csv", index=False)
pd.DataFrame(cross_rows).round(4).to_csv("data/decode_cross.csv", index=False)

we're friends . -> nous sommes amies .   calls: {'encoder': 1, 'decoder': 5}
 step 1 input: ['<start>'] -> [('nous', 0.999), ('on', 0.001), ('nos', 0.0), ('notre', 0.0), ('sommes', 0.0)]
 step 2 input: ['<start>', 'nous'] -> [('sommes', 1.0), ('nous', 0.0), ('amis', 0.0), ('avons', 0.0), ('voulons', 0.0)]
 step 3 input: ['<start>', 'nous', 'sommes'] -> [('amies', 0.961), ('amis', 0.036), ('des', 0.002), ('amie', 0.0), ("d'amies", 0.0)]
 step 4 input: ['<start>', 'nous', 'sommes', 'amies'] -> [('.', 1.0), ('du', 0.0), ('amies', 0.0), ('?', 0.0), ('!', 0.0)]
 step 5 input: ['<start>', 'nous', 'sommes', 'amies', '.'] -> [('<end>', 1.0), ('!', 0.0), ('.', 0.0), ('"', 0.0), ('on', 0.0)]


i think you're right . -> je pense que vous avez raison .   calls: {'encoder': 1, 'decoder': 8}
 step 1 input: ['<start>'] -> [('je', 0.997), ("j'ai", 0.002), ("j'y", 0.0), ('j', 0.0), ('il', 0.0)]
 step 2 input: ['<start>', 'je'] -> [('pense', 0.989), ('crois', 0.011), ('trouve', 0.0), ('prends', 0.0), ('sais', 0.0)]
 step 3 input: ['<start>', 'je', 'pense'] -> [('que', 0.999), ("qu'il", 0.001), ('dire', 0.0), ("qu'on", 0.0), ('avoir', 0.0)]
 step 4 input: ['<start>', 'je', 'pense', 'que'] -> [('vous', 0.67), ('tu', 0.327), ("t'es", 0.001), ('ça', 0.001), ("c'est", 0.0)]
 step 5 input: ['<start>', 'je', 'pense', 'que', 'vous'] -> [('avez', 0.958), ('êtes', 0.025), ('as', 0.014), ('en', 0.001), ("l'avez", 0.0)]
 step 6 input: ['<start>', 'je', 'pense', 'que', 'vous', 'avez'] -> [('raison', 1.0), ('bien', 0.0), ('tort', 0.0), ('à', 0.0), ('un', 0.0)]
 step 7 input: ['<start>', 'je', 'pense', 'que', 'vous', 'avez', 'raison'] -> [('.', 1.0), ('à', 0.0), ('?', 0.0), ('de', 0.0), ('bien', 0

## 5. Checks
(a) One teacher-forced pass over <start> + the generated words gives, at every position, the same probabilities as
the step-by-step loop gave at that step.

In [7]:
demo = "i think you're right .".split()
out, tgt, steps = translate([demo], keep=True)
src = encode([demo], en_ix, LE)
enc = model.encode(src)
logits_all = model.decode(tgt[:, :-1], enc, src)[0]
p_all = tf.nn.softmax(logits_all).numpy()[0]
diff = max(np.abs(p_all[t] - steps[t]["p"]).max() for t in range(len(steps)))
print("largest difference, one pass vs step by step:", diff)
assert diff < 1e-4

largest difference, one pass vs step by step: 6.556511e-07


(b) With the mask, the decoder's output vectors at earlier positions are the same at every step: adding a word at
the end does not change them. Without the mask they change.

In [8]:
for causal in (True, False):
    changes = []
    for t in range(2, tgt.shape[1]):
        _, y_prev, _ = model.decode(tgt[:, :t - 1], enc, src, causal=causal)
        _, y_now, _ = model.decode(tgt[:, :t], enc, src, causal=causal)
        changes.append(float(np.abs(y_now.numpy()[0, :t - 1] - y_prev.numpy()[0]).max()))
    print("mask", causal, ": largest change of earlier positions' outputs per step:", np.round(changes, 6))

mask True : largest change of earlier positions' outputs per step: [0. 0. 0. 0. 0. 0. 0.]


mask False : largest change of earlier positions' outputs per step: [11.893  7.646  6.04   7.761  9.044  4.49   5.551]


(c) Inference without the mask: the model was trained with it. Same greedy loop, mask switched off.

In [9]:
hyp_nomask = translate_all(test_en, causal=False)
same = np.mean([a == b for a, b in zip(hyp, hyp_nomask)])
print("seed 0: test BLEU with mask:", round(bleu(hyp, test_refs), 1), " without mask:",
      round(bleu(hyp_nomask, test_refs), 1), " identical translations:", round(100 * same, 1), "%")
for e, a, b in list(zip(test_en, hyp, hyp_nomask))[:40]:
    if a != b:
        print(" ".join(e), "| with mask:", " ".join(a), "| without:", " ".join(b))
mask_rows = [dict(seed=0, bleu_with=bleu(hyp, test_refs), bleu_without=bleu(hyp_nomask, test_refs), identical=same)]

seed 0: test BLEU with mask: 41.4  without mask: 38.9  identical translations: 57.4 %
that doesn't help me . | with mask: ça ne m'aide pas . | without: ça ne m'aide .
she accompanied her friend to the concert . | with mask: elle a accompagné de son ami . | without: elle était accompagné au concert .
what's his most recent novel ? | with mask: qu'est - ce qui a le plus de <unk> ? | without: qu'est - ce qui a la plupart des <unk> ?
i have nothing to do with it . | with mask: je n'ai rien à faire avec le faire . | without: je n'ai rien à le faire avec moi .
he stood aside for her to enter . | with mask: il se tenait de côté où se pénétrer . | without: il se tenait de côté où aller .
everyone was horrified . | with mask: tout le monde était visiblement horrifié . | without: tout le monde a été horrifié .
look up the word in the dictionary . | with mask: regardez ce mot en va . | without: regardez ce mot .
they are alone . | with mask: ils sont seule . | without: ils sont seuls .
i had to g

## 6. Cross-attention weights for one sentence pair (for the cross-attention Note)
One teacher-forced pass over <start> + the model's own translation; the last decoder block's cross-attention
weights, per head and averaged over the 4 heads. Rows: French positions (the word each position predicts);
columns: English words.

In [10]:
examples = []
for sent in ("we're friends .", "i think you're right .", "she is reading a book ."):
    e = sent.split()
    out, tgt, _ = translate([e], keep=True)
    src = encode([e], en_ix, LE)
    _, _, cross = model.decode(tgt[:, :-1], model.encode(src), src)
    w = cross[-1].numpy()[0][:, :, :len(e)]                     # heads x French positions x English words
    examples.append(dict(english=e, french_input=[fr_w[k] for k in tgt[0, :-1]],
                         french_output=[fr_w[k] for k in tgt[0, 1:]],
                         weights_mean=np.round(w.mean(0), 4).tolist(), weights_per_head=np.round(w, 4).tolist(),
                         note="last decoder block of a 2-block transformer (d_model 128, 4 heads) trained in "
                              "1084-transformer-inference; rows: French positions, columns: English words"))
    print(" ".join(e), "->", " ".join(examples[-1]["french_output"]))
    print(pd.DataFrame(w.mean(0), index=examples[-1]["french_output"], columns=e).round(2))

we're friends . -> nous sommes amies . <end>
        we're  friends     .
nous     0.78     0.21  0.01
sommes   0.62     0.34  0.04
amies    0.00     0.92  0.08
.        0.05     0.40  0.55
<end>    0.11     0.05  0.83


i think you're right . -> je pense que vous avez raison . <end>
           i  think  you're  right     .
je      0.61   0.38    0.00   0.00  0.00
pense   0.01   0.99    0.00   0.00  0.00
que     0.06   0.18    0.42   0.16  0.18
vous    0.01   0.09    0.84   0.03  0.03
avez    0.00   0.05    0.46   0.41  0.08
raison  0.00   0.01    0.08   0.81  0.11
.       0.03   0.03    0.04   0.16  0.73
<end>   0.03   0.02    0.20   0.15  0.60


she is reading a book . -> elle lit un livre . <end>
        she    is  reading     a  book     .
elle   0.77  0.09     0.12  0.01  0.01  0.01
lit    0.02  0.31     0.58  0.03  0.02  0.04
un     0.00  0.16     0.08  0.46  0.04  0.26
livre  0.00  0.00     0.06  0.02  0.90  0.01
.      0.03  0.23     0.12  0.04  0.02  0.55
<end>  0.04  0.18     0.06  0.15  0.12  0.44


## 7. Mask at inference, two more seeds
The same model trained from two other random starts; greedy decoding of the test set with and without the mask.

In [11]:
for seed in (1, 2):
    model = build(seed)
    model.fit((Xe, Xd), Yd, batch_size=128, epochs=25, validation_split=0.05, verbose=0)
    a, b = translate_all(test_en), translate_all(test_en, causal=False)
    mask_rows.append(dict(seed=seed, bleu_with=bleu(a, test_refs), bleu_without=bleu(b, test_refs),
                          identical=np.mean([x == y for x, y in zip(a, b)])))
    print(mask_rows[-1])
mask_df = pd.DataFrame(mask_rows)
print(mask_df.round(3))
print("mean over 3 seeds:", mask_df.mean().round(3).to_dict())
mask_df.round(4).to_csv("data/mask_at_inference.csv", index=False)

{'seed': 1, 'bleu_with': np.float64(40.7718520197157), 'bleu_without': np.float64(38.414450641314254), 'identical': np.float64(0.574)}


{'seed': 2, 'bleu_with': np.float64(42.3313232114511), 'bleu_without': np.float64(39.394404134236765), 'identical': np.float64(0.584)}
   seed  bleu_with  bleu_without  identical
0     0     41.431        38.931      0.574
1     1     40.772        38.414      0.574
2     2     42.331        39.394      0.584
mean over 3 seeds: {'seed': 1.0, 'bleu_with': 41.512, 'bleu_without': 38.913, 'identical': 0.577}


## 8. Data for two animations in other Notes
The model of section 3, trained again from the same random start (seed 0) with the same settings. GPU training is
not repeatable to the last bit, so its numbers can differ slightly from sections 3 to 6. Two exports:

(a) The masked self-attention of the first decoder block on the decoder input "<start> comment ça va ?" (source
"how are you ?"), per head: raw scores q·k, scaled scores, masked weights. For the dot-grid animation of the masked
self-attention Note.

(b) "we're friends ." through the whole model: the encoder's vectors and self-attention weights per block, then every
greedy decoder step (masked self-attention and cross-attention of the predicted position, the 5 most likely words).
For the end-to-end animation of the transformer end-to-end Note.

In [5]:
model = build(0)
model.fit((Xe, Xd), Yd, batch_size=128, epochs=25, validation_split=0.05, verbose=0)
hyp0 = translate_all(test_en)
print("retrained seed 0: test BLEU", round(bleu(hyp0, test_refs), 1))


def mha_parts(layer, xq, xk, causal=False, kmask=None):
    """Raw scores, scaled scores, weights and output of a Keras MultiHeadAttention layer, by hand (no dropout)."""
    Wq, bq, Wk, bk, Wv, bv, Wo, bo = [w.numpy() for w in layer.weights]
    q = np.einsum("nd,dhk->hnk", xq, Wq) + bq[:, None]
    k = np.einsum("nd,dhk->hnk", xk, Wk) + bk[:, None]
    v = np.einsum("nd,dhk->hnk", xk, Wv) + bv[:, None]
    raw = q @ k.transpose(0, 2, 1)
    scaled = raw / np.sqrt(q.shape[-1])
    s = scaled.copy()
    if causal:
        s = s + np.triu(np.full(s.shape[1:], -np.inf), 1)
    if kmask is not None:
        s = np.where(kmask[None, None, :], s, -np.inf)
    w = np.exp(s - s.max(-1, keepdims=True))
    w /= w.sum(-1, keepdims=True)
    out = np.einsum("hnk,hkd->nd", w @ v, Wo) + bo
    return raw, scaled, w, out

retrained seed 0: test BLEU 40.6


In [6]:
src_words, dec_words = "how are you ?".split(), ["<start>"] + "comment ça va ?".split()
assert all(w in en_ix for w in src_words) and all(w in fr_ix for w in dec_words)
src = encode([src_words], en_ix, LE)
tgt = np.array([[fr_ix[w] for w in dec_words]], "int32")
y0 = model.embed(model.emb_fr, tgt, False).numpy()[0]
raw, scaled, w, _ = mha_parts(model.dec_blocks[0].self_mha, y0, y0, causal=True)
_, w_keras = model.dec_blocks[0].self_mha(y0[None], y0[None], use_causal_mask=True, return_attention_scores=True)
print("largest difference from Keras' weights:", np.abs(w - w_keras.numpy()[0]).max())
assert np.allclose(w, w_keras.numpy()[0], atol=1e-5)
rows = [dict(head=h + 1, query=dec_words[i], key=dec_words[j], i=i + 1, j=j + 1, raw=raw[h, i, j],
             scaled=scaled[h, i, j], weight=w[h, i, j]) for h in range(HEADS) for i in range(5) for j in range(5)]
grid = pd.DataFrame(rows)
grid.round(4).to_csv("../1081-masked-self-attention/data/trained_grid.csv", index=False)
for h in range(HEADS):
    print(f"head {h + 1}: raw scores\n", pd.DataFrame(raw[h], dec_words, dec_words).round(1))
    print("masked weights\n", pd.DataFrame(w[h], dec_words, dec_words).round(2))

largest difference from Keras' weights: 3.1683411583216525e-07
head 1: raw scores
          <start>    comment    ça         va     ?
<start>     -0.6   2.600000   1.9   1.500000   2.6
comment      1.7 -19.700001  -9.7  -2.600000  17.9
ça          -0.7  -0.100000 -14.9   1.000000  10.8
va           3.1 -11.400000  -5.9 -17.299999  -1.1
?            0.4   0.600000  27.5 -29.700001 -74.0
masked weights
          <start>  comment    ça    va    ?
<start>     1.00     0.00  0.00  0.00  0.0
comment     0.98     0.02  0.00  0.00  0.0
ça          0.46     0.51  0.04  0.00  0.0
va          0.76     0.06  0.16  0.02  0.0
?           0.01     0.01  0.98  0.00  0.0
head 2: raw scores
          <start>    comment    ça         va          ?
<start>     -0.1  -0.000000  -1.0  -1.400000  -0.700000
comment      4.0 -17.200001 -16.6 -21.600000   3.200000
ça           0.1   7.800000  -3.9   3.500000  17.200001
va           0.4 -10.300000 -13.1 -39.900002   5.200000
?            8.7 -31.200001  -0.9 -68

In [7]:
def encode_parts(src_ids):
    """The encoder by hand, block by block: the input vectors, each block's mean-head weights and output."""
    x = model.embed(model.emb_en, src_ids, False).numpy()[0]
    keep = src_ids[0] > 0
    xs, ws = [x], []
    for b in model.enc_blocks:
        _, _, w, z = mha_parts(b.mha, x, x, kmask=keep)
        x1 = b.n1(x + z).numpy()
        x = b.n2(x1 + b.f2(b.f1(x1))).numpy()
        xs.append(x)
        ws.append(w.mean(0))
    return xs, ws


demo = "we're friends .".split()
src = encode([demo], en_ix, LE)
xs, ws = encode_parts(src)
n = len(demo)
print("encoder by hand vs model.encode:", np.abs(xs[-1][:n] - model.encode(src).numpy()[0][:n]).max())
assert np.allclose(xs[-1][:n], model.encode(src).numpy()[0][:n], atol=1e-4)
enc = model.encode(src)
tgt = np.array([[START]], "int32")
steps = []
while True:
    y = model.embed(model.emb_fr, tgt, False).numpy()[0]
    selfw, crossw = [], []
    for b in model.dec_blocks:
        _, _, w1, z = mha_parts(b.self_mha, y, y, causal=True)
        y1 = b.n1(y + z).numpy()
        _, _, w2, zc = mha_parts(b.cross_mha, y1, enc.numpy()[0], kmask=src[0] > 0)
        y2 = b.n2(y1 + zc).numpy()
        y = b.n3(y2 + b.f2(b.f1(y2))).numpy()
        selfw.append(w1.mean(0)[-1].tolist())
        crossw.append(w2.mean(0)[-1, :n].tolist())
    p = tf.nn.softmax(model.out(y[-1:])).numpy()[0]
    p_model = tf.nn.softmax(model.decode(tgt, enc, src, last_only=True)[0][0, -1]).numpy()
    assert np.abs(p - p_model).max() < 1e-4
    top = p.argsort()[::-1][:5]
    steps.append(dict(input=[fr_w[k] for k in tgt[0]], top=[[fr_w[k], round(float(p[k]), 4)] for k in top],
                      self_last=[np.round(r, 4).tolist() for r in selfw],
                      cross_last=[np.round(r, 4).tolist() for r in crossw]))
    nxt = int(top[0])
    tgt = np.concatenate([tgt, [[nxt]]], 1)
    print("step", len(steps), steps[-1]["input"], "->", steps[-1]["top"][:3])
    if nxt == END or len(steps) == LF - 1:
        break
cap = dict(english=demo, d_model=D, heads=HEADS, blocks=NBLOCKS,
           enc_vectors=[np.round(x[:n], 3).tolist() for x in xs],
           enc_weights=[np.round(w[:n, :n], 4).tolist() for w in ws], steps=steps,
           note="trained 2+2-block transformer of 1084-transformer-inference (seed 0, retrained in section 8); "
                "weights are means over the 4 heads; self_last/cross_last: the row of the newest position, per decoder block")
json.dump(cap, open("../1085-transformer-end-to-end/data/capstone.json", "w"))
print("translation:", " ".join(s["top"][0][0] for s in steps))

encoder by hand vs model.encode: 2.3841858e-07
step 1 ['<start>'] -> [['nous', 0.9959], ['on', 0.0025], ['nos', 0.0006]]


step 2 ['<start>', 'nous'] -> [['sommes', 0.9997], ['nous', 0.0001], ['avons', 0.0]]
step 3 ['<start>', 'nous', 'sommes'] -> [['amies', 0.9868], ['amis', 0.011], ['des', 0.0018]]
step 4 ['<start>', 'nous', 'sommes', 'amies'] -> [['.', 0.9985], ['du', 0.0013], ['des', 0.0]]
step 5 ['<start>', 'nous', 'sommes', 'amies', '.'] -> [['<end>', 1.0], ['.', 0.0], ['!', 0.0]]
translation: nous sommes amies . <end>
